# Week 3 — Expand Dataset + Build Data Loader
**Urdu OCR Project — Code Saviours (SMC-Private) Limited, Batch SI-26**

This week has two parts:
1. Grow the dataset from 100 images to 200+ images.
2. Build a PyTorch `Dataset` class so the model can actually load and use that data.

Think of the `Dataset` class as the kitchen. It does not cook the model itself, but nothing gets cooked without it.


## Step 1: Expand to 200+ images

This part cannot be done from inside a notebook, since it means going back to your sources (newspapers, books, signboards, synthetic text, handwriting) and collecting more images by hand. It just is what it is.

What the cell below does instead is give you a quick way to check your progress as you add images: how many you have, how many more you need, whether any files are missing, and whether the duplicate-row bug from Week 1 has crept back in.

Run it any time after adding a batch of new images to `labels.csv`.


In [ ]:
import pandas as pd
import os

# This points at your existing Week 1 data folder.
# Change it if you moved your data into a new SI26-Week3 folder instead.
DATA_DIR = "/workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week1/data"
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")

df = pd.read_csv(LABELS_PATH)

print(f"Total images in labels.csv: {len(df)}")
print()

# Breakdown by category (newspaper, book, signboard, synthetic, other)
if "category" in df.columns:
    print("Images per category:")
    print(df["category"].value_counts())
else:
    print("No 'category' column found — check your CSV headers.")

print()
target = 200
current = len(df)
remaining = max(0, target - current)
print(f"You need {remaining} more image(s) to reach the 200 minimum.")

# Check that every file listed in the CSV actually exists on disk
missing = []
for _, row in df.iterrows():
    img_path = row["image"]
    full_path = img_path if os.path.isabs(img_path) else os.path.join(DATA_DIR, img_path)
    if not os.path.exists(full_path):
        missing.append(img_path)

if missing:
    print(f"\nWarning: {len(missing)} image file(s) in labels.csv are missing from disk:")
    for m in missing[:10]:
        print(" -", m)
    if len(missing) > 10:
        print(f"   ...and {len(missing) - 10} more")
else:
    print("\nAll image files listed in labels.csv were found on disk.")

# Check for duplicate rows (same bug that showed up in Week 1)
dupes = df[df.duplicated(subset=["image"], keep=False)]
if len(dupes) > 0:
    print(f"\nWarning: {len(dupes)} duplicate row(s) found for the same image.")
else:
    print("\nNo duplicate rows found.")


## Step 2: Build a Dataset Class

A `Dataset` class is code that tells your model: "here is one image, and here is the correct text for it." It works like a filing system. Whenever the model asks for sample number N, the class opens the right image, processes it, and hands both the image and its label back.

PyTorch requires two methods on every dataset:
- `__len__` — returns how many samples there are in total
- `__getitem__` — returns one sample, given its index

### Cell 1 — Install and import


In [ ]:
!pip install transformers torch pillow pandas


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os


class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, base_dir=""):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.base_dir = base_dir
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Build the full image path. This works whether your CSV stores
        # full paths or paths relative to the data folder (e.g. "raw/books/0001.jpg").
        img_path = row["image"]
        if self.base_dir and not os.path.isabs(img_path):
            img_path = os.path.join(self.base_dir, img_path)

        # Load and convert image
        image = Image.open(img_path).convert("RGB")

        # Process image for the model
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {"pixel_values": pixel_values, "labels": labels}


### Cell 2 — Test your dataset

This loads the TrOCR processor, builds the dataset from your `labels.csv`, checks that a sample loads correctly, and splits the data into training and testing sets.


In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

dataset = UrduOCRDataset(LABELS_PATH, processor, base_dir=DATA_DIR)

# Test it loads correctly
sample = dataset[0]
print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")


## Before you submit

- [ ] `labels.csv` has 200+ rows, all pointing to real image files
- [ ] Ran Cell 1 (dataset audit) with no missing files and no duplicates
- [ ] Ran Week 2 preprocessing on all new images so they land in `data/processed/`
- [ ] Pushed the updated `labels.csv` and this notebook to GitHub
- [ ] Commented on the submission with: *"My dataset has X images and loads correctly"*

**Note:** `microsoft/trocr-base-printed` is trained on English text, so accuracy on Urdu will be rough for now — that's expected at this stage. It's just here to prove the pipeline (image in, tensor out) works. A Urdu-capable checkpoint is a Week 4+ problem.
